# 1) Imports

In [1]:
import json
import sys
import os
import pandas as pd
import random
import torch
from torch.utils.data import DataLoader

# Get the absolute path to the project root
project_root = os.path.abspath(os.path.join(os.path.abspath('..'), '..'))

# Add correct paths
models_path = os.path.join(project_root, 'models')
src_path = os.path.join(project_root, 'src')
data_path = os.path.join(project_root, 'data')

# Append to sys.path
sys.path.append(models_path)
sys.path.append(src_path)
sys.path.append(data_path)

from data_loader_helper import SequenceDataset, collate_batch
from xLSTM.dkt_k_fold import k_fold_cv_dkt
from xLSTM.dkt_train import train_dkt
from KTDataset import KTDataset


# 2) Model Training and Evaluation

## 2.1) Data imports and transforms


In [2]:
data_path = os.path.abspath(os.path.join('..', '..', 'data', 'preprocessed', 'df_answers.csv'))
df_answers = pd.read_csv(data_path)

skill_names_path = os.path.abspath(os.path.join('..', '..', 'data', 'preprocessed', 'df_skill_names.csv'))
df_skill_names = pd.read_csv(skill_names_path)

additional_columns = None
#additional_columns = ['ease']
#additional_columns = ['ease', 'ms_first_response', 'bottom_hint']

kt_dataset = KTDataset(df_answers, df_skill_names, prepare_DKT=True, additional_columns=additional_columns)

user_dict = kt_dataset.DKT_datadict
num_skills = kt_dataset.num_skills
num_other = kt_dataset.num_other


## 2.2) Seting constants

In [3]:
train_ratio = 0.8  # 80% for training, 20% for testing
NUM_EPOCHS = 15
BATCH_SIZE = 100
NUM_FOLDS = 5  # Number of folds for cross-validation
HID_SIZE = 200

embed_dim = 3
special_embed = "tanh"

finetune = False


In [4]:
if additional_columns is None:
    model_version = "basic"
elif len(additional_columns) == 1:
    model_version = "ease"
else:
    model_version = "full"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2.3) Splitting data into train and test sets

In [5]:
# Get all keys and shuffle them
keys = list(user_dict.keys())
random.shuffle(keys)

# Split keys into train and test
split_index = int(len(keys) * train_ratio)
train_keys = keys[:split_index]
test_keys = keys[split_index:]

# Create train and test dictionaries
train_dict = {key: user_dict[key] for key in train_keys}
test_dict = {key: user_dict[key] for key in test_keys}


## 2.4) Hyperparameter-tuning

In [6]:
if finetune:
  configs = [
    {
        "learning_rate": lr,
        "model_params": {
            "num_skills": num_skills,
            "num_other": num_other,
            "embed_dim": embed_dim,
            "hid_size": HID_SIZE,
            "num_hid_layers": num_hid_layers,
            "drop_prob": drop_prob,
            "special_embed": special_embed
        },
    }
    for lr in [1e-3, 1e-4, 1e-5]  # 3 reasonable options for learning rate
    for num_hid_layers in [1, 2, 3]  # Hidden layers 1 or 2
    for drop_prob in [0.3, 0.4, 0.5]  # Dropout rate 0.3, 0.4, 0.5
    ]

  best_val_auc_avg = 0  # Track the best validation AUC
  best_config = None  # Track the best configuration

  for idx, config in enumerate(configs, 1):
      log_message = f"\nEvaluating Config {idx}/{len(configs)}"
      print(log_message)  # Print to console

      lr = config['learning_rate']
      model_params = config['model_params']

      val_auc_avg = k_fold_cv_dkt(
          num_folds=NUM_FOLDS,
          model_params=model_params,
          lr=lr,
          num_epochs=NUM_EPOCHS,
          device=device,
          train_dict=train_dict,
          batch_size=BATCH_SIZE,
          )

      # Update the global best if needed
      if val_auc_avg > best_val_auc_avg:
          best_val_auc_avg = val_auc_avg
          best_config = config  # Save the best configuration
          best_message = f"\nNew best model found: Config {idx}. Validation AUC: {best_val_auc_avg:.4f}"
          print(best_message)  # Print to console

  # Final output
  final_message = f"\nBest test AUC: {best_val_auc_avg:.4f}"
  print(final_message)

  final_config_message = f"\nBest Configuration: {best_config}"
  print(final_config_message)

else:
  # Load the configuration from the JSON file
  with open('xlstm_best_config.json', 'r') as f:
      best_config = json.load(f)

## 2.5) Model training

In [ ]:
# Training on the whole train set
train_dict = dict(sorted(train_dict.items(), key=lambda item: len(item[1])))
train_dataset = SequenceDataset(train_dict)
train_loader = DataLoader(train_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=False, num_workers=2)

test_dict = dict(sorted(test_dict.items(), key=lambda item: len(item[1])))
test_dataset = SequenceDataset(test_dict)
test_loader = DataLoader(test_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=False, num_workers=2)

# Log training details
training_message = "Training on the whole train set"
print(training_message)  # Print to console

best_model, list_val_loss, list_val_auc = train_dkt(
    model_params=best_config['model_params'],
    lr=best_config['learning_rate'],
    num_epochs=NUM_EPOCHS,
    device=device,
    train_loader=train_loader
)


Training on the whole train set

Epoch 1

Training:   0%|          | 0/30 [00:00<?, ? batches/s]

In [25]:
best_config = {
    "learning_rate": 0.001,
    "model_params": {
        "num_skills": 109,
        "num_other": 1,
        "embed_dim": 3,
        "hid_size": 200,
        "num_hid_layers": 2,
        "drop_prob": 0.4,
        "special_embed": "tanh"
    }
}

## 2.6) Model evaluation

In [50]:
train_loss, train_auc = process(best_model, train_loader, device)
test_loss, test_auc = process(best_model, test_loader, device)

# Example results for a model
results = {
    'Dataset': ['Train', 'Test'],
    'AUC': [train_auc, test_auc],
    'Loss': [train_loss, test_loss]
}

# Convert to DataFrame
df = pd.DataFrame(results)

print(df)


Evaluation: 100%|██████████| 8/8 [00:01<00:00,  4.91 batches/s]
  Dataset       AUC      Loss
0   Train  0.713485  0.547942
1    Test  0.713036  0.444262


In [54]:
train_loss, train_auc = process(best_model, train_loader, device)
test_loss, test_auc = process(best_model, test_loader, device)

# Example results for a model
results = {
    'Dataset': ['Train', 'Test'],
    'AUC': [train_auc, test_auc],
    'Loss': [train_loss, test_loss]
}

# Convert to DataFrame
df = pd.DataFrame(results)

print(df)


Evaluation: 100%|██████████| 8/8 [00:01<00:00,  5.91 batches/s]
  Dataset       AUC      Loss
0   Train  0.714369  0.546642
1    Test  0.711428  0.452316


## 2.7) Saving the results

In [ ]:
# Saving the model
model_save_path = f'dkt_{model_version}_model.pth'
torch.save(best_model.state_dict(), model_save_path)
model_save_message = f"Model saved to {model_save_path}"

print(model_save_message)  # Print to console

# Save as CSV
df.to_csv(f'dkt_{model_version}_model_results.csv', index=False)
